# MATH GR5360 Final Project - Notebook 3
## Performance Metrics & Extended Analysis

**Columbia University - Mathematical Methods in Financial Price Analysis**

---

## ⚠️ CONFIGURE YOUR GROUP'S MARKET BELOW ⚠️

In [ ]:

PRIMARY_MARKET= 1  # <-- UPDATE: Your group number (1-18)

# ============================================================================

In [ ]:
# Market database
MARKET_DATABASE = {
    'BO': ('Soybean Oil', 'CBOT-CME', 600, 39, 6, 1),
    'DX': ('Dollar Index', 'NYBOT-ICE', 1000, 16.5, 5, 1),
    'HG': ('Copper', 'COMEX-NYMEX-CME', 250, 59.25, 12.5, 1),
    'HO': ('Heating Oil', 'NYMEX-CME', 420, 70.2, 4.2, 100),
    'JO': ('Orange Juice', 'NYBOT-ICE', 150, 183, 7.5, 1),
    'JY': ('Japanese Yen', 'CME', 1250, 53, 6.25, 100),
    'SY': ('Soybeans', 'CBOT-CME', 50, 35.5, 12.5, 1),
    'SB': ('Sugar #11', 'NYBOT-ICE', 1120, 56.76, 11.2, 1),
    'SF': ('Swiss Franc', 'CME', 1250, 25.5, 12.5, 100),
    'TU': ('2-Year Treasury', 'CBOT-CME', 2000, 18.625, 15.625, 1),
    'TY': ('10-Year Treasury', 'CBOT-CME', 1000, 18.625, 15.625, 1),
    'WC': ('Wheat', 'CBOT-CME', 50, 30.5, 12.5, 1),
    'SM': ('Soybean Meal', 'CBOT-CME', 100, 57, 10, 1),
    'CC': ('Cocoa', 'NYBOT-ICE', 10, 103, 10, 1),
    'BZ': ('Schatz', 'EUREX', 1000, 10.5, 5, 1),
    'CL': ('Crude Oil WTI', 'NYMEX-CME', 1000, 46, 10, 1),
    'GC': ('Gold 100oz', 'COMEX-NYMEX-CME', 100, 65, 10, 1),
    'SV': ('Silver', 'COMEX-NYMEX-CME', 5000, 243, 25, 0.01),
}

GROUP_TO_TICKER = {
    1: 'BO', 2: 'DX', 3: 'HG', 4: 'HO', 5: 'JO', 6: 'JY',
    7: 'SY', 8: 'SB', 9: 'SF', 10: 'TU', 11: 'TY', 12: 'WC',
    13: 'SM', 14: 'CC', 15: 'BZ', 16: 'CL', 17: 'GC', 18: 'SV'
}

TICKER = GROUP_TO_TICKER[GROUP_NUMBER]
market_info = MARKET_DATABASE[TICKER]
MARKET = {
    'ticker': TICKER, 'name': market_info[0], 'exchange': market_info[1],
    'PV': market_info[2], 'slpg': market_info[3], 'E0': 100000,
}

print(f"GROUP {GROUP_NUMBER}: {TICKER} - {MARKET['name']}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import jit
from tqdm.notebook import tqdm
from typing import Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

---
## 1. Load Results & Strategy

In [ ]:
# Load walk-forward results
try:
    wf_results = pd.read_csv(f'../results/{TICKER}_walkforward.csv')
    print(f"Loaded {len(wf_results)} walk-forward periods")
except FileNotFoundError:
    print("Run Notebook 02 first to generate walk-forward results.")
    wf_results = None

In [ ]:
# Strategy function (same as Notebook 02)
@jit(nopython=True, cache=True)
def channel_strategy(Open, High, Low, Close, L, S, slpg, PV, E0, barsBack):
    N = len(Close)
    E = np.zeros(N) + E0
    DD = np.zeros(N)
    trades = np.zeros(N)
    HH = np.zeros(N)
    LL = np.zeros(N)
    for k in range(barsBack, N):
        HH[k] = np.max(High[k-L:k])
        LL[k] = np.min(Low[k-L:k])
    position = 0
    benchmarkLong = 0.0
    benchmarkShort = 0.0
    Emax = E0
    for k in range(barsBack, N):
        traded = False
        delta = PV * (Close[k] - Close[k-1]) * position
        if position == 0:
            buy = High[k] >= HH[k]
            sell = Low[k] <= LL[k]
            if buy and sell:
                delta = -slpg + PV * (LL[k] - HH[k])
                trades[k] = 1
            else:
                if buy:
                    delta = -slpg/2 + PV * (Close[k] - HH[k])
                    position = 1
                    traded = True
                    benchmarkLong = High[k]
                    trades[k] = 0.5
                if sell:
                    delta = -slpg/2 - PV * (Close[k] - LL[k])
                    position = -1
                    traded = True
                    benchmarkShort = Low[k]
                    trades[k] = 0.5
        elif position == 1 and not traded:
            sellShort = Low[k] <= LL[k]
            sell = Low[k] <= (benchmarkLong * (1 - S))
            if sellShort and sell:
                delta = delta - slpg - 2 * PV * (Close[k] - LL[k])
                position = -1
                benchmarkShort = Low[k]
                trades[k] = 1
            else:
                if sell:
                    delta = delta - slpg/2 - PV * (Close[k] - (benchmarkLong * (1 - S)))
                    position = 0
                    trades[k] = 0.5
                if sellShort:
                    delta = delta - slpg - 2 * PV * (Close[k] - LL[k])
                    position = -1
                    benchmarkShort = Low[k]
                    trades[k] = 1
            benchmarkLong = max(High[k], benchmarkLong)
        elif position == -1 and not traded:
            buyLong = High[k] >= HH[k]
            buy = High[k] >= (benchmarkShort * (1 + S))
            if buyLong and buy:
                delta = delta - slpg + 2 * PV * (Close[k] - HH[k])
                position = 1
                benchmarkLong = High[k]
                trades[k] = 1
            else:
                if buy:
                    delta = delta - slpg/2 + PV * (Close[k] - (benchmarkShort * (1 + S)))
                    position = 0
                    trades[k] = 0.5
                if buyLong:
                    delta = delta - slpg + 2 * PV * (Close[k] - HH[k])
                    position = 1
                    benchmarkLong = High[k]
                    trades[k] = 1
            benchmarkShort = min(Low[k], benchmarkShort)
        E[k] = E[k-1] + delta
        Emax = max(Emax, E[k])
        DD[k] = E[k] - Emax
    return E, DD, trades

In [ ]:
# Load data
DATA_FILE = f'../data/{TICKER}-5min.csv'

def load_data(filepath):
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip().str.lower()
    if 'date' in df.columns and 'time' in df.columns:
        df['datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['time'].astype(str), format='mixed')
    else:
        df['datetime'] = pd.to_datetime(df.iloc[:, 0])
    df.set_index('datetime', inplace=True)
    df.sort_index(inplace=True)
    rename = {}
    for c in df.columns:
        if 'open' in c.lower(): rename[c] = 'Open'
        elif 'high' in c.lower(): rename[c] = 'High'
        elif 'low' in c.lower(): rename[c] = 'Low'
        elif 'close' in c.lower(): rename[c] = 'Close'
    df.rename(columns=rename, inplace=True)
    return df[['Open', 'High', 'Low', 'Close']].astype(float)

try:
    df = load_data(DATA_FILE)
    print(f"Loaded {len(df):,} bars")
except FileNotFoundError:
    # Generate synthetic
    np.random.seed(42 + GROUP_NUMBER)
    n = 250000
    dates = pd.date_range('2008-01-01', periods=n, freq='5min')
    start_prices = {'BO': 30, 'DX': 90, 'HG': 3, 'HO': 2, 'JO': 100, 'JY': 0.009,
                    'SY': 900, 'SB': 15, 'SF': 1, 'TU': 105, 'TY': 115, 'WC': 500,
                    'SM': 300, 'CC': 2500, 'BZ': 110, 'CL': 60, 'GC': 1300, 'SV': 15}
    start_price = start_prices.get(TICKER, 100)
    returns = np.random.randn(n) * 0.0003 + np.sin(np.linspace(0, 8*np.pi, n)) * 0.0001
    close = start_price * np.exp(np.cumsum(returns))
    df = pd.DataFrame({
        'Open': close * (1 + np.random.randn(n) * 0.0001),
        'High': close * (1 + np.abs(np.random.randn(n) * 0.0003)),
        'Low': close * (1 - np.abs(np.random.randn(n) * 0.0003)),
        'Close': close
    }, index=dates)
    df['High'] = df[['Open', 'High', 'Close']].max(axis=1)
    df['Low'] = df[['Open', 'Low', 'Close']].min(axis=1)
    print(f"Generated {len(df):,} synthetic bars")

---
## 2. Performance Metrics (as required by assignment)

In [ ]:
def calculate_metrics(equity, trades, E0=100000, bars_per_year=19656):
    """Calculate all required performance metrics."""
    final = equity[-1]
    profit = final - E0
    
    # Returns
    equity_nz = equity[equity > 0]
    returns = np.diff(equity_nz) / equity_nz[:-1]
    
    avg_ret = np.mean(returns)
    std_ret = np.std(returns)
    ann_ret = avg_ret * bars_per_year
    ann_std = std_ret * np.sqrt(bars_per_year)
    sharpe = ann_ret / ann_std if ann_std > 0 else 0
    
    # Drawdown
    peak = np.maximum.accumulate(equity)
    dd = equity - peak
    max_dd = dd.min()
    max_dd_pct = max_dd / peak[np.argmin(dd)] if peak[np.argmin(dd)] > 0 else 0
    
    # Trades
    pnl = np.diff(equity)
    trade_mask = trades[1:] > 0
    trade_pnl = pnl[trade_mask] if np.any(trade_mask) else np.array([0])
    
    n_trades = int(np.sum(trades))
    winners = trade_pnl[trade_pnl > 0]
    losers = trade_pnl[trade_pnl < 0]
    
    win_rate = len(winners) / max(len(winners) + len(losers), 1)
    avg_win = np.mean(winners) if len(winners) > 0 else 0
    avg_loss = np.mean(losers) if len(losers) > 0 else 0
    
    gross_profit = np.sum(winners) if len(winners) > 0 else 0
    gross_loss = -np.sum(losers) if len(losers) > 0 else 1
    profit_factor = gross_profit / max(gross_loss, 1)
    
    roa = profit / abs(max_dd) if max_dd != 0 else 0
    
    return {
        'Total Profit': profit,
        'Return %': (final / E0 - 1) * 100,
        'Ann. Return': ann_ret * 100,
        'Ann. Volatility': ann_std * 100,
        'Sharpe Ratio': sharpe,
        'Max Drawdown $': max_dd,
        'Max Drawdown %': max_dd_pct * 100,
        'Return on Account': roa,
        'Total Trades': n_trades,
        'Win Rate %': win_rate * 100,
        'Avg Winner': avg_win,
        'Avg Loser': avg_loss,
        'Win/Loss Ratio': abs(avg_win / avg_loss) if avg_loss != 0 else 0,
        'Profit Factor': profit_factor,
    }

In [ ]:
# Get best parameters from walk-forward (or use default)
if wf_results is not None and len(wf_results) > 0:
    best_L = int(wf_results['L'].mode().iloc[0])
    best_S = float(wf_results['S'].mode().iloc[0])
else:
    best_L, best_S = 5000, 0.02

print(f"Using parameters: L={best_L}, S={best_S}")

# Run full backtest
barsBack = max(best_L + 1, 100)
E, DD, trades = channel_strategy(
    df['Open'].values, df['High'].values, df['Low'].values, df['Close'].values,
    best_L, best_S, MARKET['slpg'], MARKET['PV'], MARKET['E0'], barsBack
)

metrics = calculate_metrics(E, trades, MARKET['E0'])

In [ ]:
# Display metrics
print("\n" + "=" * 60)
print(f"PERFORMANCE METRICS: {TICKER} (L={best_L}, S={best_S})")
print("=" * 60)

print("\n📈 RETURNS:")
print(f"   Total Profit:    ${metrics['Total Profit']:>12,.2f}")
print(f"   Total Return:    {metrics['Return %']:>12.2f}%")
print(f"   Ann. Return:     {metrics['Ann. Return']:>12.4f}%")
print(f"   Ann. Volatility: {metrics['Ann. Volatility']:>12.4f}%")

print("\n📊 RISK-ADJUSTED:")
print(f"   Sharpe Ratio:      {metrics['Sharpe Ratio']:>12.4f}")
print(f"   Return on Account: {metrics['Return on Account']:>12.4f}")

print("\n📉 DRAWDOWN:")
print(f"   Max Drawdown $:  ${metrics['Max Drawdown $']:>12,.2f}")
print(f"   Max Drawdown %:  {metrics['Max Drawdown %']:>12.2f}%")

print("\n🎯 TRADE STATISTICS:")
print(f"   Total Trades:    {metrics['Total Trades']:>12.0f}")
print(f"   Win Rate:        {metrics['Win Rate %']:>12.2f}%")
print(f"   Avg Winner:      ${metrics['Avg Winner']:>12,.2f}")
print(f"   Avg Loser:       ${metrics['Avg Loser']:>12,.2f}")
print(f"   Win/Loss Ratio:  {metrics['Win/Loss Ratio']:>12.4f}")
print(f"   Profit Factor:   {metrics['Profit Factor']:>12.4f}")

print("\n" + "=" * 60)

In [ ]:
# Save metrics
pd.DataFrame([metrics]).to_csv(f'../results/{TICKER}_performance_metrics.csv', index=False)
print(f"Saved to ../results/{TICKER}_performance_metrics.csv")

---
## 3. Extended Analysis: Varying T and τ

In [ ]:
def run_wf_single(df, L_vals, S_vals, T, tau, slpg, PV, E0):
    """Run walk-forward with specific T and τ."""
    total_bars = len(df)
    total_days = (df.index.max() - df.index.min()).days
    bars_per_day = total_bars / total_days if total_days > 0 else 78
    bars_per_year = int(bars_per_day * 252)
    bars_per_quarter = int(bars_per_year / 4)
    
    IS_bars = T * bars_per_year
    OOS_bars = tau * bars_per_quarter
    
    results = []
    idx = 0
    
    while idx + IS_bars + OOS_bars <= total_bars:
        # IS optimization (simplified)
        best_obj = -np.inf
        best_is_profit = 0
        best_L, best_S = L_vals[0], S_vals[0]
        
        for L in L_vals:
            for S in S_vals:
                barsBack = max(int(L) + 1, 100)
                if IS_bars < barsBack + 100: continue
                E, DD, tr = channel_strategy(
                    df['Open'].values[idx:idx+IS_bars],
                    df['High'].values[idx:idx+IS_bars],
                    df['Low'].values[idx:idx+IS_bars],
                    df['Close'].values[idx:idx+IS_bars],
                    int(L), float(S), slpg, PV, E0, barsBack
                )
                profit = E[-1] - E[barsBack]
                max_dd = DD.min()
                obj = profit / abs(max_dd) if max_dd != 0 else 0
                if obj > best_obj:
                    best_obj = obj
                    best_is_profit = profit
                    best_L, best_S = int(L), float(S)
        
        # OOS
        barsBack = max(best_L + 1, 100)
        oos_start = idx + IS_bars
        oos_end = oos_start + OOS_bars
        if oos_end - oos_start < barsBack + 100:
            idx += OOS_bars
            continue
        
        E, DD, tr = channel_strategy(
            df['Open'].values[oos_start:oos_end],
            df['High'].values[oos_start:oos_end],
            df['Low'].values[oos_start:oos_end],
            df['Close'].values[oos_start:oos_end],
            best_L, best_S, slpg, PV, E0, barsBack
        )
        oos_profit = E[-1] - E[barsBack]
        oos_dd = DD.min()
        oos_obj = oos_profit / abs(oos_dd) if oos_dd != 0 else 0
        
        results.append({'is_obj': best_obj, 'oos_obj': oos_obj, 'oos_profit': oos_profit})
        idx += OOS_bars
    
    if len(results) == 0:
        return {'T': T, 'tau': tau, 'error': True}
    
    avg_is = np.mean([r['is_obj'] for r in results])
    avg_oos = np.mean([r['oos_obj'] for r in results])
    total_oos = np.sum([r['oos_profit'] for r in results])
    decay = avg_oos / avg_is if avg_is > 0 else 0
    
    return {'T': T, 'tau': tau, 'n': len(results), 'avg_is': avg_is, 'avg_oos': avg_oos, 'decay': decay, 'total_oos': total_oos, 'error': False}

In [ ]:
# Run extended analysis
L_vals = np.array([2000, 4000, 6000, 8000])
S_vals = np.array([0.01, 0.02, 0.03, 0.04])
T_values = [1, 2, 3, 4, 5, 6]
tau_values = [1, 2, 3, 4]

print("Running extended analysis (varying T and τ)...")
ext_results = []

for T in tqdm(T_values, desc="T"):
    for tau in tau_values:
        res = run_wf_single(df, L_vals, S_vals, T, tau, MARKET['slpg'], MARKET['PV'], MARKET['E0'])
        ext_results.append(res)

ext_df = pd.DataFrame(ext_results)
valid = ext_df[~ext_df['error']]
print(f"\nCompleted {len(valid)} valid combinations")

In [ ]:
# Display extended analysis
if len(valid) > 0:
    print("\n" + "=" * 60)
    print("EXTENDED ANALYSIS: T vs τ")
    print("=" * 60)
    print(valid[['T', 'tau', 'n', 'avg_is', 'avg_oos', 'decay', 'total_oos']].to_string(index=False))
    
    # Best by decay
    best = valid.loc[valid['decay'].idxmax()]
    print(f"\n📊 Best Decay: T={int(best['T'])}yr, τ={int(best['tau'])}Q (decay={best['decay']:.1%})")

In [ ]:
# Visualize
if len(valid) > 0:
    pivot = valid.pivot(index='T', columns='tau', values='decay')
    
    plt.figure(figsize=(10, 6))
    plt.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    plt.colorbar(label='Decay (OOS/IS)')
    plt.xticks(range(len(pivot.columns)), [f"{t}Q" for t in pivot.columns])
    plt.yticks(range(len(pivot.index)), [f"{t}Y" for t in pivot.index])
    plt.xlabel('τ (OOS quarters)')
    plt.ylabel('T (IS years)')
    plt.title(f'{TICKER}: Performance Decay by T and τ', fontweight='bold')
    
    # Annotate
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.iloc[i, j]
            if not np.isnan(val):
                plt.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(f'../results/06_{TICKER}_extended_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Save extended analysis
if len(valid) > 0:
    valid.to_csv(f'../results/{TICKER}_extended_analysis.csv', index=False)
    print(f"Saved to ../results/{TICKER}_extended_analysis.csv")

---
## 4. Final Summary

In [ ]:
print("\n" + "=" * 70)
print(f"FINAL SUMMARY: {TICKER} ({MARKET['name']})")
print("=" * 70)

print(f"\n1. Strategy: Channel WithDDControl (L={best_L}, S={best_S})")
print(f"2. Total Profit: ${metrics['Total Profit']:,.2f}")
print(f"3. Sharpe Ratio: {metrics['Sharpe Ratio']:.4f}")
print(f"4. Max Drawdown: ${abs(metrics['Max Drawdown $']):,.2f}")
print(f"5. Win Rate: {metrics['Win Rate %']:.1f}%")
print(f"6. Profit Factor: {metrics['Profit Factor']:.2f}")

if len(valid) > 0:
    best = valid.loc[valid['decay'].idxmax()]
    print(f"\n7. Recommended T: {int(best['T'])} years")
    print(f"8. Recommended τ: {int(best['tau'])} quarter(s)")
    print(f"9. Expected Decay: {best['decay']:.1%}")

print("\n" + "=" * 70)

---
## Next Steps

1. **Secondary Market**: Choose from Bitcoin (#19) or Chinese futures (#20-25)
2. **C Implementation**: Port strategy for performance
3. **Presentation**: Create 30-40 min PowerPoint

**End of Notebook 03**